In [34]:
from __future__ import annotations

import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.tree import DecisionTreeRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
import joblib
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

from pathlib import Path
import sys

# notebooks/ -> project_root/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# добавляем корень репо в sys.path
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("app exists:", (PROJECT_ROOT / "app").exists())

from app.core.transformers import (
    RegionGrouper,
    BinaryToFloat,
    ATMLocationFeatures,
)

print("RegionGrouper module:", RegionGrouper.__module__)

PROJECT_ROOT = C:\Users\steel\Documents\GitHub\GeoATM-popularity
app exists: True
RegionGrouper module: app.core.transformers


In [35]:
# Загрузка датасета
DATASET_URL = "https://raw.githubusercontent.com/Khamoon7/GeoATM-popularity/refs/heads/main/data/train_data.csv"
data = pd.read_csv(DATASET_URL)
print(f"Размер: {data.shape}")
data.head()

Размер: (6200, 48)


,id,atm_group,address_raw,address_geocoded,geo_lon,geo_lat,country,region,municipality,city,...,nearest_public_transport_dist_m,count_public_transport_300m,nearest_parking_dist_m,count_parking_300m,nearest_education_dist_m,count_education_300m,nearest_subway_dist_m,nearest_post_offices_dist_m,count_post_offices_300m,has_subway_nearby
0,5.0,496.5,BUDENNOGO 7A ELISTA,"Россия, Республика Калмыкия, Элиста, улица С.М...",44.260605,46.318231,Россия,Республика Калмыкия,городской округ Элиста,Элиста,...,93.6,5,143.7,3,247.3,1,0.0,0.0,0,False
1,6.0,496.5,"HO CHI MIHN AVE, 19 ULYANOVSK","Россия, Ульяновск, проспект Хо Ши Мина, 19",48.300652,54.270443,Россия,Ульяновская область,городской округ Ульяновск,Ульяновск,...,89.9,6,NaN,0,260.8,1,0.0,220.4,2,False
2,7.0,496.5,SHELESTA 116A KHABAROVSK,"Россия, Хабаровск, улица Шелеста, 116А",135.052594,48.520497,Россия,Хабаровский край,городской округ Хабаровск,Хабаровск,...,33.8,8,112.9,6,0.0,0,0.0,186.6,2,False
3,8.0,496.5,ORDZHONIKIDZE 52 YAKUTSK,"Россия, Республика Саха (Якутия), Якутск, улиц...",129.721308,62.025566,Россия,Республика Саха (Якутия),городской округ Якутск,Якутск,...,119.8,7,246.9,4,195.4,5,0.0,167.2,1,False
4,10.0,496.5,"VETERANOV AVE, 3 KRASNOKAMENS","Россия, Забайкальский край, Краснокаменск, про...",118.027480,50.090714,Россия,Забайкальский край,Краснокаменский муниципальный округ,Краснокаменск,...,70.6,3,48.3,5,NaN,0,NaN,NaN,0,False


## EDA 

In [36]:
# Конфигурация на основе датасета
EXCLUDE_COLS = [
    "id", "address_raw", "address_geocoded", "street", "house", 
    "geo_lon", "geo_lat", "municipality", "country", "atm_group" # ПОКА ВЫКИНУЛА atm_group из обучения 
]

BIN_FEATURES = [
    "is_24_7", "contactless_tech", "qr_codes", "usd_available", "eur_available",
    "cash_in", "cash_out", "cashless_pay", "account_statement", 
    "access_for_disabled", "transfer_p2p", "transfer_a2a", "loan_payments", 
    "has_subway_nearby"
]

LOCATION_FEATURES = [
    "population_density_per_km2", "nearest_malls_dist_m", "count_malls_300m",
    "nearest_supermarkets_dist_m", "nearest_pharmacies_hospitals_dist_m",
    "count_pharmacies_hospitals_300m", "count_banks_atms_300m",
    "nearest_cafes_dist_m", "count_cafes_300m", "nearest_restaurants_dist_m",
    "count_restaurants_300m", "nearest_public_transport_dist_m",
    "count_public_transport_300m", "nearest_parking_dist_m", "count_parking_300m",
    "nearest_education_dist_m", "count_education_300m", "nearest_subway_dist_m",
    "nearest_post_offices_dist_m"
]

CATEGORICAL_FEATURES = ["city"]
TARGET = "target"

print(f"Бинарных фичей: {len(BIN_FEATURES)}")
print(f"Локационных фичей: {len(LOCATION_FEATURES)}")


Бинарных фичей: 14
Локационных фичей: 19


## INPUT DATASET vs STANDART COLUMNS FOR PYPLINE

In [37]:
# Доступные колонки из наших списков
available_location = [col for col in LOCATION_FEATURES if col in data.columns]
available_bin = [col for col in BIN_FEATURES if col in data.columns]
available_cat = [col for col in CATEGORICAL_FEATURES if col in data.columns]

print(f"\nДоступно:")
print(f"   Локационные: {len(available_location)}/{len(LOCATION_FEATURES)}")
print(f"   Бинарные:     {len(available_bin)}/{len(BIN_FEATURES)}")
print(f"   Категории:    {len(available_cat)}/{len(CATEGORICAL_FEATURES)}")

# Безопасная подготовка X
exclude_cols_present = [col for col in EXCLUDE_COLS if col in data.columns]
X = data.drop(columns=exclude_cols_present + [TARGET], errors='ignore')


Доступно:
   Локационные: 19/19
   Бинарные:     14/14
   Категории:    1/1


## Preprocessor


In [38]:
def create_preprocessor(X: pd.DataFrame):
    all_cols = X.columns.tolist()

    available_num = [col for col in LOCATION_FEATURES if col in all_cols]
    available_bin = [col for col in BIN_FEATURES if col in all_cols]
    available_cat = [col for col in CATEGORICAL_FEATURES if col in all_cols]

    print("НАЙДЕНЫ:")
    print(f"  num: {len(available_num)}")
    print(f"  bin: {len(available_bin)}")
    print(f"  cat: {len(available_cat)}")

    initial_pipeline = Pipeline(steps=[
        ("region_grouper", RegionGrouper()),
        ("binary_to_float", BinaryToFloat()),
        ("location_features", ATMLocationFeatures()),
    ])

    transformers = [
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]), available_num),

        ("bin", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
        ]), available_bin),
    ]

    # добавляем cat ТОЛЬКО если колонка реально есть
    if available_cat:
        transformers.append((
            "cat",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
            ]),
            available_cat,
        ))

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

    full_pipeline = Pipeline(steps=[
        ("initial", initial_pipeline),
        ("preprocessor", preprocessor),
        ("final_scaler", StandardScaler()),
    ])

    return full_pipeline


full_pipeline = create_preprocessor(X)

НАЙДЕНЫ:
  num: 19
  bin: 14
  cat: 1


## Pypline Preprocessor Validate and Safe

In [39]:
print("Тест:")
X_processed = full_pipeline.fit_transform(X[:100])
print(f"Shape: {X_processed.shape}")
print(f"NA: {np.isnan(X_processed).sum()}")
print(f"Finite: {np.isfinite(X_processed).all()}")

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

save_path = MODELS_DIR / "preprocessor.pkl"
joblib.dump(full_pipeline, save_path)
print("✅ СОХРАНЕНО:", save_path)
print("Размер файла:", save_path.stat().st_size / 1024, "KB")

Тест:
Shape: (100, 108)
NA: 0
Finite: True
✅ СОХРАНЕНО: C:\Users\steel\Documents\GitHub\GeoATM-popularity\models\preprocessor.pkl
Размер файла: 11.861328125 KB


## Train

In [40]:
X = data.drop(columns=["target"], errors="ignore")
y = data["target"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# =========================
# 2) ЗАГРУЗКА PREPROCESSOR
# =========================
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

preproc_path = MODELS_DIR / "preprocessor.pkl"
preprocessor = joblib.load(preproc_path)
print("Loaded preprocessor:", preproc_path)

X_train shape: (4650, 47)
X_test shape: (1550, 47)
Loaded preprocessor: C:\Users\steel\Documents\GitHub\GeoATM-popularity\models\preprocessor.pkl


In [41]:
models = {
    "LinearRegression": LinearRegression(),
    "LassoCV": LassoCV(cv=5, random_state=42, n_jobs=-1),
    "RidgeCV": RidgeCV(cv=5),
    "DecisionTree": DecisionTreeRegressor(random_state=42, max_depth=10),
}

results = {}

for name, base_model in models.items():
    preprocessor = joblib.load(preproc_path)

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),  # работает по сырым X
        ("model", base_model),
    ])

    cv_scores = cross_val_score(
        pipe,
        X_train,   # СЫРЫЕ признаки
        y_train,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,  # ✅ ускорение
    )

    results[name] = {
        "CV_RMSE_mean": float(-cv_scores.mean()),
        "CV_RMSE_std": float(cv_scores.std()),
        "CV_scores": cv_scores,
    }
    

In [33]:
# Таблица результатов
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('CV_RMSE_mean')
results_df.rename(columns={
    'CV_RMSE_mean': 'rmse_mean',
    'CV_RMSE_std': 'rmse_std'
}, inplace=True)
display(results_df)

best_model_name = results_df.index[0]
print(f"ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")

best_base_model = models[best_model_name]
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('models', best_base_model),
])

# Обучаем на train
best_pipeline.fit(X_train, y_train)

# Оценка на test
y_pred_test = best_pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred_test)
test_rmse = np.sqrt(mse)

print(f"\n{best_model_name}")
print(f"   CV RMSE (mean): {results_df.loc[best_model_name, 'rmse_mean']:.4f}")
print(f"   CV RMSE (std):  {results_df.loc[best_model_name, 'rmse_std']:.4f}")
print(f"   Test RMSE:      {test_rmse:.4f}")


,rmse_mean,rmse_std,CV_scores
LinearRegression,NaN,NaN,"[nan, nan, nan, nan, nan]"
LassoCV,NaN,NaN,"[nan, nan, nan, nan, nan]"
RidgeCV,NaN,NaN,"[nan, nan, nan, nan, nan]"
DecisionTree,NaN,NaN,"[nan, nan, nan, nan, nan]"


ЛУЧШАЯ МОДЕЛЬ: LinearRegression


NotFittedError: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [13]:
test_results = {}

for name, base_model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('models', base_model),
    ])
    
    # Обучаем на train
    pipe.fit(X_train, y_train)
    
    # Предсказываем на test
    y_pred = pipe.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    test_results[name] = {
        'test_rmse': rmse
    }
    print(f"{name:15} | Test RMSE: {rmse:.4f}")

test_results_df = pd.DataFrame(test_results).T.sort_values('test_rmse')
display(test_results_df)


/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


LinearRegression | Test RMSE: 0.0715


/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


LassoCV         | Test RMSE: 0.0705


/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


RidgeCV         | Test RMSE: 0.0712
DecisionTree    | Test RMSE: 0.0684


/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/shon/opt/anaconda3/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


,test_rmse
DecisionTree,0.068410
LassoCV,0.070531
RidgeCV,0.071160
LinearRegression,0.071503


In [12]:
# ===============================
# FINAL ATM PIPELINE (ONE BLOCK)
# ===============================

# контроль: чтобы pickle не ссылался на __main__
assert RegionGrouper.__module__ == "app.core.transformers", RegionGrouper.__module__
assert BinaryToFloat.__module__ == "app.core.transformers", BinaryToFloat.__module__
assert ATMLocationFeatures.__module__ == "app.core.transformers", ATMLocationFeatures.__module__

BIN_FEATURES = [
    "is_24_7", "contactless_tech", "qr_codes", "usd_available", "eur_available",
    "cash_in", "cash_out", "cashless_pay", "account_statement",
    "access_for_disabled", "transfer_p2p", "transfer_a2a",
    "loan_payments", "has_subway_nearby"
]

LOCATION_FEATURES = [
    "population_density_per_km2", "nearest_malls_dist_m", "count_malls_300m",
    "nearest_supermarkets_dist_m", "nearest_pharmacies_hospitals_dist_m",
    "count_pharmacies_hospitals_300m", "count_banks_atms_300m",
    "nearest_cafes_dist_m", "count_cafes_300m",
    "nearest_restaurants_dist_m", "count_restaurants_300m",
    "nearest_public_transport_dist_m", "count_public_transport_300m",
    "nearest_parking_dist_m", "count_parking_300m",
    "nearest_education_dist_m", "count_education_300m",
    "nearest_subway_dist_m", "nearest_post_offices_dist_m"
]

CATEGORICAL_FEATURES = ["city"]

TARGET = "target"

# сохраняем в /models относительно корня репо
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS_DIR / "final_atm_pipeline.pkl"


# ===============================
# PREPROCESSOR
# ===============================

def build_preprocessor(X_df: pd.DataFrame) -> Pipeline:
    available_num = [c for c in LOCATION_FEATURES if c in X_df.columns]
    available_bin = [c for c in BIN_FEATURES if c in X_df.columns]
    available_cat = [c for c in CATEGORICAL_FEATURES if c in X_df.columns]

    column_transformer = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
            ]), available_num),

            ("bin", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
            ]), available_bin),

            ("cat", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(
                    drop="first",
                    sparse_output=False,
                    handle_unknown="ignore"
                )),
            ]), available_cat),
        ],
        remainder="drop",
    )

    return Pipeline(steps=[
        ("region_grouper", RegionGrouper()),        # использует app.core.transformers
        ("binary_cast", BinaryToFloat()),           # использует app.core.transformers
        ("location_features", ATMLocationFeatures()),# использует app.core.transformers
        ("columns", column_transformer),
        ("scaler", StandardScaler()),
    ])


# ===============================
# FINAL PIPELINE
# ===============================

preprocessor = build_preprocessor(X)

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("models", best_base_model),   # LassoCV — лучшая модель
])

# ===============================
# TRAIN ON FULL DATA
# ===============================

final_pipeline.fit(X, y)

y_pred = final_pipeline.predict(X)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"RMSE (FULL DATA): {rmse:.4f}")

# ===============================
# SAVE
# ===============================

joblib.dump(final_pipeline, MODEL_PATH)
print(f"FINAL PIPELINE SAVED TO: {MODEL_PATH}")


NameError: name 'best_base_model' is not defined

In [42]:
from __future__ import annotations

import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42

# notebooks/ -> project_root/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("app exists:", (PROJECT_ROOT / "app").exists())

from app.core.transformers import RegionGrouper, BinaryToFloat, ATMLocationFeatures

print("RegionGrouper module:", RegionGrouper.__module__)

# =========================
# DATA
# =========================
DATASET_URL = "https://raw.githubusercontent.com/Khamoon7/GeoATM-popularity/refs/heads/main/data/train_data.csv"
data = pd.read_csv(DATASET_URL)
print("data shape:", data.shape)

EXCLUDE_COLS = [
    "id", "address_raw", "address_geocoded", "street", "house",
    "geo_lon", "geo_lat", "municipality", "country", "atm_group",
]

BIN_FEATURES = [
    "is_24_7", "contactless_tech", "qr_codes", "usd_available", "eur_available",
    "cash_in", "cash_out", "cashless_pay", "account_statement",
    "access_for_disabled", "transfer_p2p", "transfer_a2a", "loan_payments",
    "has_subway_nearby",
]

LOCATION_FEATURES = [
    "population_density_per_km2", "nearest_malls_dist_m", "count_malls_300m",
    "nearest_supermarkets_dist_m", "nearest_pharmacies_hospitals_dist_m",
    "count_pharmacies_hospitals_300m", "count_banks_atms_300m",
    "nearest_cafes_dist_m", "count_cafes_300m", "nearest_restaurants_dist_m",
    "count_restaurants_300m", "nearest_public_transport_dist_m",
    "count_public_transport_300m", "nearest_parking_dist_m", "count_parking_300m",
    "nearest_education_dist_m", "count_education_300m", "nearest_subway_dist_m",
    "nearest_post_offices_dist_m",
]

CATEGORICAL_FEATURES = ["city"]
TARGET = "target"

# ✅ ВАЖНО: X ВЕЗДЕ ОДИНАКОВЫЙ (без мусорных колонок)
exclude_cols_present = [c for c in EXCLUDE_COLS if c in data.columns]
X = data.drop(columns=exclude_cols_present + [TARGET], errors="ignore")
y = data[TARGET].astype(float)

print("X shape:", X.shape, "y shape:", y.shape)

# =========================
# PREPROCESSOR FACTORY
# =========================
def build_preprocessor(X_df: pd.DataFrame) -> Pipeline:
    cols = X_df.columns.tolist()
    available_num = [c for c in LOCATION_FEATURES if c in cols]
    available_bin = [c for c in BIN_FEATURES if c in cols]
    available_cat = [c for c in CATEGORICAL_FEATURES if c in cols]

    print("FOUND:")
    print("  num:", len(available_num))
    print("  bin:", len(available_bin))
    print("  cat:", len(available_cat))

    transformers = [
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]), available_num),

        ("bin", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
        ]), available_bin),
    ]

    if available_cat:
        transformers.append((
            "cat",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
            ]),
            available_cat,
        ))

    column_transformer = ColumnTransformer(transformers=transformers, remainder="drop")

    return Pipeline(steps=[
        ("region_grouper", RegionGrouper()),
        ("binary_cast", BinaryToFloat()),
        ("location_features", ATMLocationFeatures()),
        ("columns", column_transformer),
        ("scaler", StandardScaler()),
    ])

# =========================
# QUICK SANITY CHECK
# =========================
preproc = build_preprocessor(X)
Xt = preproc.fit_transform(X.head(200))
print("Sanity transform shape:", Xt.shape)
print("NaNs:", np.isnan(Xt).sum(), "finite:", np.isfinite(Xt).all())

# =========================
# SAVE PREPROCESSOR (OPTIONAL)
# =========================
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
preproc_path = MODELS_DIR / "preprocessor.pkl"
joblib.dump(preproc, preproc_path)
print("✅ Saved preprocessor:", preproc_path)

# =========================
# TRAIN/TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

# =========================
# CV MODELS
# =========================
models = {
    "LinearRegression": LinearRegression(),
    "LassoCV": LassoCV(cv=5, random_state=RANDOM_STATE, n_jobs=-1),
    "RidgeCV": RidgeCV(cv=5),
    "DecisionTree": DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=10),
}

results = {}

for name, base_model in models.items():
    # ✅ каждый раз новый preprocessor (чтобы CV клонировался чисто)
    preproc_cv = build_preprocessor(X_train)

    pipe = Pipeline(steps=[
        ("preprocessor", preproc_cv),
        ("model", base_model),
    ])

    cv_scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        error_score="raise",  # ✅ НЕ скрываем ошибки (иначе NaN)
    )

    results[name] = {
        "rmse_mean": float(-cv_scores.mean()),
        "rmse_std": float(cv_scores.std()),
        "cv_scores": cv_scores,
    }

results_df = pd.DataFrame(results).T.sort_values("rmse_mean")
display(results_df)

best_model_name = results_df.index[0]
print("ЛУЧШАЯ МОДЕЛЬ:", best_model_name)

# =========================
# BEST PIPELINE FIT + TEST
# =========================
best_base_model = models[best_model_name]
best_preproc = build_preprocessor(X_train)

best_pipeline = Pipeline(steps=[
    ("preprocessor", best_preproc),
    ("model", best_base_model),
])

best_pipeline.fit(X_train, y_train)

y_pred_test = best_pipeline.predict(X_test)
test_rmse = float(np.sqrt(mean_squared_error(y_test, y_pred_test)))

print(f"\n{best_model_name}")
print(f"   CV RMSE (mean): {results_df.loc[best_model_name, 'rmse_mean']:.4f}")
print(f"   CV RMSE (std):  {results_df.loc[best_model_name, 'rmse_std']:.4f}")
print(f"   Test RMSE:      {test_rmse:.4f}")

# =========================
# FINAL PIPELINE TRAIN ON FULL DATA + SAVE
# =========================
# контроль: чтобы pickle не ссылался на __main__
assert RegionGrouper.__module__ == "app.core.transformers", RegionGrouper.__module__
assert BinaryToFloat.__module__ == "app.core.transformers", BinaryToFloat.__module__
assert ATMLocationFeatures.__module__ == "app.core.transformers", ATMLocationFeatures.__module__

final_preproc = build_preprocessor(X)
final_pipeline = Pipeline(steps=[
    ("preprocessor", final_preproc),
    ("model", best_base_model),
])

final_pipeline.fit(X, y)

final_path = MODELS_DIR / "final_atm_pipeline.pkl"
joblib.dump(final_pipeline, final_path)
print("✅ FINAL PIPELINE SAVED TO:", final_path)


PROJECT_ROOT = C:\Users\steel\Documents\GitHub\GeoATM-popularity
app exists: True
RegionGrouper module: app.core.transformers
data shape: (6200, 48)
X shape: (6200, 37) y shape: (6200,)
FOUND:
  num: 19
  bin: 14
  cat: 1
Sanity transform shape: (200, 117)
NaNs: 0 finite: True
✅ Saved preprocessor: C:\Users\steel\Documents\GitHub\GeoATM-popularity\models\preprocessor.pkl
FOUND:
  num: 19
  bin: 14
  cat: 1
FOUND:
  num: 19
  bin: 14
  cat: 1
FOUND:
  num: 19
  bin: 14
  cat: 1
FOUND:
  num: 19
  bin: 14
  cat: 1


,rmse_mean,rmse_std,cv_scores
LassoCV,0.067343,0.001136,"[-0.0694765466341922, -0.06618560807355559, -0..."
RidgeCV,0.068855,0.001319,"[-0.07139369408656017, -0.0678531178082435, -0..."
DecisionTree,0.069624,0.002529,"[-0.07374320483498667, -0.06837773525934059, -..."
LinearRegression,0.086183,0.032952,"[-0.07257390574300193, -0.15201825169837013, -..."


ЛУЧШАЯ МОДЕЛЬ: LassoCV
FOUND:
  num: 19
  bin: 14
  cat: 1


C:\Users\steel\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)



LassoCV
   CV RMSE (mean): 0.0673
   CV RMSE (std):  0.0011
   Test RMSE:      0.0705
FOUND:
  num: 19
  bin: 14
  cat: 1
✅ FINAL PIPELINE SAVED TO: C:\Users\steel\Documents\GitHub\GeoATM-popularity\models\final_atm_pipeline.pkl


In [43]:
from pathlib import Path
import joblib
import pandas as pd

# 1) путь к пайплайну
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = PROJECT_ROOT / "models" / "final_atm_pipeline.pkl"
model = joblib.load(MODEL_PATH)

print("Loaded:", MODEL_PATH)
print("Steps:", list(model.named_steps.keys()))

# 2) у нас шаг называется "model" (не "models")
print("Model step type:", type(model.named_steps["model"]))

# 3) sample ДОЛЖЕН содержать только те колонки, что были в X при обучении
# (т.е. без EXCLUDE_COLS + без target)
sample = pd.DataFrame([{
    "region": "Республика Калмыкия",   # если region реально был в X
    "city": "Элиста",

    "population_density_per_km2": 3.52,

    "is_24_7": False,
    "contactless_tech": False,
    "qr_codes": False,
    "usd_available": False,
    "eur_available": False,
    "cash_in": True,
    "cash_out": True,
    "cashless_pay": False,
    "account_statement": True,
    "access_for_disabled": True,
    "transfer_p2p": True,
    "transfer_a2a": False,
    "loan_payments": False,
    "has_subway_nearby": False,

    "nearest_malls_dist_m": 928.7,
    "count_malls_300m": 0,
    "nearest_supermarkets_dist_m": 364.3,
    "nearest_pharmacies_hospitals_dist_m": 242.6,
    "count_pharmacies_hospitals_300m": 2,
    "count_banks_atms_300m": 2,
    "nearest_cafes_dist_m": 124.5,
    "count_cafes_300m": 1,
    "nearest_restaurants_dist_m": 317.9,
    "count_restaurants_300m": 0,
    "nearest_public_transport_dist_m": 93.6,
    "count_public_transport_300m": 5,
    "nearest_parking_dist_m": 143.7,
    "count_parking_300m": 3,
    "nearest_education_dist_m": 247.3,
    "count_education_300m": 1,
    "nearest_subway_dist_m": 0.0,
    "nearest_post_offices_dist_m": 0.0,
}])

pred = model.predict(sample)
print("pred:", float(pred[0]))


Loaded: C:\Users\steel\Documents\GitHub\GeoATM-popularity\models\final_atm_pipeline.pkl
Steps: ['preprocessor', 'model']
Model step type: <class 'sklearn.linear_model._coordinate_descent.LassoCV'>
pred: -0.01298421215010773


In [3]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

m = joblib.load("final_atm_pipeline.pkl")

print("MODEL TYPE:", type(m))

def find_column_transformers(obj, path="model"):
    found = []
    if isinstance(obj, ColumnTransformer):
        found.append((path, obj))
    if isinstance(obj, Pipeline):
        for step_name, step in obj.steps:
            found += find_column_transformers(step, f"{path}.{step_name}")
    return found

cts = find_column_transformers(m)

if not cts:
    print("❌ ColumnTransformer не найден внутри пайплайна. Тогда ожидания по колонкам надо искать в другом месте.")
else:
    for path, ct in cts:
        print("\n=== ColumnTransformer at:", path, "===")
        cols_all = []
        for name, transformer, cols in ct.transformers:
            # cols обычно list[str] или str, иногда callable/slice
            print(f"- block: {name:20s} | transformer={type(transformer)} | cols_type={type(cols)}")
            if isinstance(cols, (list, tuple)):
                cols_all.extend([c for c in cols if isinstance(c, str)])
            elif isinstance(cols, str):
                cols_all.append(cols)
            else:
                print("  ⚠️ cols is not list/str ->", cols)

        cols_all = sorted(set(cols_all))
        print("\nEXPECTED INPUT COLS (from ColumnTransformer):")
        for c in cols_all:
            print(" ", c)


MODEL TYPE: <class 'sklearn.pipeline.Pipeline'>

=== ColumnTransformer at: model.preprocessor.columns ===
- block: num                  | transformer=<class 'sklearn.pipeline.Pipeline'> | cols_type=<class 'list'>
- block: bin                  | transformer=<class 'sklearn.pipeline.Pipeline'> | cols_type=<class 'list'>
- block: cat                  | transformer=<class 'sklearn.pipeline.Pipeline'> | cols_type=<class 'list'>

EXPECTED INPUT COLS (from ColumnTransformer):
  access_for_disabled
  account_statement
  cash_in
  cash_out
  cashless_pay
  city
  contactless_tech
  count_banks_atms_300m
  count_cafes_300m
  count_education_300m
  count_malls_300m
  count_parking_300m
  count_pharmacies_hospitals_300m
  count_public_transport_300m
  count_restaurants_300m
  eur_available
  has_subway_nearby
  is_24_7
  loan_payments
  nearest_cafes_dist_m
  nearest_education_dist_m
  nearest_malls_dist_m
  nearest_parking_dist_m
  nearest_pharmacies_hospitals_dist_m
  nearest_post_offices_dist_m